# PointNet++ SemanticKITTI Fine-Tuning & Testing
This notebook copies your SemanticKITTI dataset from Google Drive to Colab's local high-speed disk, sets up the PointNet++ repository, runs the fine-tuning process, and finishes by running a test to visualize the Ground Truth vs Prediction!

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Copy the dataset to Colab's local disk for fast I/O
!mkdir -p /content/dataset_trimmed
print("Copying dataset from Drive... This may take a moment depending on the size.")
!cp -r "/content/drive/MyDrive/dataset_trimmed/sequences" /content/dataset_trimmed/
print("Copy complete!")

In [ ]:
# 3. Setup PointNet++ Repository
import os
if not os.path.exists('/content/Pointnet_Pointnet2_pytorch'):
    !git clone https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git
    
!pip install -q pypcd4 pandas matplotlib tqdm

In [ ]:
%%writefile semantickitti_dataset.py
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset

class SemanticKITTIDataset(Dataset):
    def __init__(self, data_path, sequences=['00'], num_points=4096, split='train'):
        self.data_path = data_path
        self.num_points = num_points
        self.split = split
        
        # 0 through 19 classes total (20 distinct values)
        self.learning_map = {
            0 : 0, 1 : 0, 10: 1, 11: 2, 13: 5, 15: 3, 16: 5, 18: 4, 20: 5, 
            30: 6, 31: 7, 32: 8, 40: 9, 44: 10, 48: 11, 49: 12, 50: 13, 
            51: 14, 52: 0, 60: 9, 70: 15, 71: 16, 72: 17, 80: 18, 81: 19, 
            99: 0, 252: 1, 253: 7, 254: 6, 255: 8, 256: 5, 257: 5, 258: 4, 259: 5
        }
        self.learning_map_lut = np.zeros((260,), dtype=np.int32)
        for k, v in self.learning_map.items():
            self.learning_map_lut[k] = v
            
        self.scan_files = []
        self.label_files = []
        
        for seq in sequences:
            scan_path = os.path.join(data_path, seq, 'velodyne')
            label_path = os.path.join(data_path, seq, 'labels')
            
            if not os.path.exists(scan_path) or not os.path.exists(label_path):
                print(f"Warning: Sequence {seq} missing velodyne or labels folder.")
                continue
                
            scans = sorted(glob.glob(os.path.join(scan_path, '*.bin')))
            labels = sorted(glob.glob(os.path.join(label_path, '*.label')))
            
            min_len = min(len(scans), len(labels))
            self.scan_files.extend(scans[:min_len])
            self.label_files.extend(labels[:min_len])
            
        print(f"Loaded {len(self.scan_files)} frames for {self.split} split.")

    def __len__(self):
        return len(self.scan_files)

    def __getitem__(self, idx):
        scan_file = self.scan_files[idx]
        label_file = self.label_files[idx]
        
        scan = np.fromfile(scan_file, dtype=np.float32).reshape((-1, 4))
        label = np.fromfile(label_file, dtype=np.uint32)
        
        sem_label = label & 0xFFFF
        sem_label = self.learning_map_lut[sem_label]
        
        num_raw = scan.shape[0]
        if num_raw >= self.num_points:
            choice = np.random.choice(num_raw, self.num_points, replace=False)
        else:
            choice = np.random.choice(num_raw, self.num_points, replace=True)
            
        sampled_scan = scan[choice, :]
        sampled_label = sem_label[choice]
        
        point_features = torch.tensor(sampled_scan, dtype=torch.float32).transpose(0, 1)
        point_labels = torch.tensor(sampled_label, dtype=torch.long)
        return point_features, point_labels


In [ ]:
%%writefile outdoor_pointnet.py
import os
import sys
import torch
import torch.nn as nn

repo_path = os.path.abspath('Pointnet_Pointnet2_pytorch')
if repo_path not in sys.path:
    sys.path.append(repo_path)

import models.pointnet2_sem_seg as pointnet2_sem_seg
from models.pointnet2_utils import PointNetSetAbstraction

def get_outdoor_model(num_classes=20, input_channels=1):
    model = pointnet2_sem_seg.get_model(num_classes)
    
    # Note: input_channels+6 because the repo passes all 3 coordinates PLUS the 4 feature channels into sa1.
    model.sa1 = PointNetSetAbstraction(1024, 0.1, 32, input_channels + 6, [32, 32, 64], False)
    model.conv2 = nn.Conv1d(128, num_classes, 1)
    return model

def load_pretrained_weights(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    pretrained_dict = checkpoint['model_state_dict']
    model_dict = model.state_dict()
    
    filtered_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)
    return model


In [ ]:
%%writefile finetune.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from semantickitti_dataset import SemanticKITTIDataset
from outdoor_pointnet import get_outdoor_model, load_pretrained_weights

def main():
    DATASET_PATH = '/content/dataset_trimmed/sequences'
    CHECKPOINT_PATH = '/content/Pointnet_Pointnet2_pytorch/log/sem_seg/pointnet2_sem_seg/checkpoints/best_model.pth'
    
    BATCH_SIZE = 4
    NUM_POINTS = 4096
    NUM_EPOCHS = 10
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 20
    INPUT_CHANNELS = 1
    
    if not os.path.exists(DATASET_PATH):
        print(f"ERROR: SemanticKITTI dataset not found at {DATASET_PATH}.")
        return

    print("Initializing datasets...")
    train_dataset = SemanticKITTIDataset(DATASET_PATH, sequences=['00'], num_points=NUM_POINTS, split='train')
    val_dataset = SemanticKITTIDataset(DATASET_PATH, sequences=['00'], num_points=NUM_POINTS, split='val')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    
    print("Initializing Outdoor PointNet++...")
    model = get_outdoor_model(num_classes=NUM_CLASSES, input_channels=INPUT_CHANNELS)
    
    if os.path.exists(CHECKPOINT_PATH):
        print("Loading pre-trained S3DIS weights...")
        model = load_pretrained_weights(model, CHECKPOINT_PATH)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    class_weights = torch.ones(NUM_CLASSES).to(device) 
    class_weights[0] = 0.0 # Ignore unlabeled
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    
    print("Starting Training Loop...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
        for batch_features, batch_labels in pbar:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            predictions, _ = model(batch_features)
            predictions = predictions.transpose(1, 2)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
            
        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Average Train Loss: {avg_train_loss:.4f}")
        
        # Save checkpoint periodically in your Drive
        os.makedirs('/content/drive/MyDrive/checkpoints_kitti', exist_ok=True)
        torch.save(model.state_dict(), f'/content/drive/MyDrive/checkpoints_kitti/semantickitti_epoch_{epoch+1}.pth')

if __name__ == '__main__':
    main()


In [ ]:
# Execute the fine-tuning script!
!python finetune.py

## Testing on Labeled Sequence (`00`)
Run this cell to test the network on a sequence that has ground truth labels, so you can compare!

In [ ]:
%matplotlib inline
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from semantickitti_dataset import SemanticKITTIDataset
from outdoor_pointnet import get_outdoor_model

def get_semantickitti_colors():
    return {
        0: [0, 0, 0],         1: [100, 150, 245],   2: [100, 230, 245],   3: [30, 60, 150],     
        4: [80, 30, 180],     5: [100, 80, 250],    6: [255, 30, 30],     7: [255, 40, 200],    
        8: [150, 30, 90],     9: [255, 0, 255],     10: [255, 150, 255],  11: [75, 0, 75],      
        12: [175, 0, 75],     13: [255, 200, 0],    14: [255, 120, 50],   15: [0, 175, 0],      
        16: [135, 60, 0],     17: [150, 240, 80],   18: [255, 240, 150],  19: [255, 0, 0]
    }

def get_semantickitti_names():
    return {
        0: 'unlabeled', 1: 'car', 2: 'bicycle', 3: 'motorcycle', 4: 'truck',
        5: 'other-vehicle', 6: 'person', 7: 'bicyclist', 8: 'motorcyclist',
        9: 'road', 10: 'parking', 11: 'sidewalk', 12: 'other-ground', 13: 'building',
        14: 'fence', 15: 'vegetation', 16: 'trunk', 17: 'terrain', 18: 'pole', 19: 'traffic-sign'
    }

DATASET_PATH = '/content/dataset_trimmed/sequences'
CHECKPOINT_PATH_10 = '/content/drive/MyDrive/checkpoints_kitti/semantickitti_epoch_10.pth'
CHECKPOINT_PATH_1 = '/content/drive/MyDrive/checkpoints_kitti/semantickitti_epoch_1.pth'

print("Loading test frame from SemanticKITTI...")
dataset = SemanticKITTIDataset(DATASET_PATH, sequences=['00'], num_points=4096, split='val')
point_features, point_labels = dataset[0]

inputs = point_features.unsqueeze(0)
labels = point_labels.numpy()
xyz = inputs[0, :3, :].transpose(0, 1).numpy()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_outdoor_model(num_classes=20, input_channels=1)

if os.path.exists(CHECKPOINT_PATH_10):
    print(f"Loading weights from {CHECKPOINT_PATH_10}...")
    model.load_state_dict(torch.load(CHECKPOINT_PATH_10, map_location=device, weights_only=False))
elif os.path.exists(CHECKPOINT_PATH_1):
    print(f"Loading weights from {CHECKPOINT_PATH_1}...")
    model.load_state_dict(torch.load(CHECKPOINT_PATH_1, map_location=device, weights_only=False))
else:
    print("ERROR: Could not find epoch 1 or epoch 10 checkpoint in your Google Drive!")

model = model.to(device)
model.eval()

print("Running inference...")
inputs = inputs.to(device)
with torch.no_grad():
    predictions, _ = model(inputs)

pred_labels = torch.argmax(predictions, dim=2).squeeze(0).cpu().numpy()

name_map = get_semantickitti_names()
unique_classes = np.unique(pred_labels)
detected_objects = [name_map.get(c, "Unknown") for c in unique_classes if c != 0]
print(f"\n---> Objects detected by PointNet++ in this frame: {', '.join(detected_objects)}\n")

print("Plotting results...")
color_map = get_semantickitti_colors()
gt_colors = np.array([color_map.get(l, [0,0,0]) for l in labels]) / 255.0
pred_colors = np.array([color_map.get(l, [0,0,0]) for l in pred_labels]) / 255.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

ax1.scatter(xyz[:, 0], xyz[:, 1], c=gt_colors, s=5, alpha=0.8)
ax1.set_title("Ground Truth", fontsize=16)
ax1.axis('equal')
ax1.set_facecolor('black') 

ax2.scatter(xyz[:, 0], xyz[:, 1], c=pred_colors, s=5, alpha=0.8)
ax2.set_title("PointNet++ Prediction", fontsize=16)
ax2.axis('equal')
ax2.set_facecolor('black')

plt.tight_layout()
plt.show()


## Testing on Unlabeled Sequence (`20`)
SemanticKITTI sequences 11-21 are for the official test set and DO NOT have `.label` files. Because they don't have labels, we cannot use the dataset class. We must load the raw `.bin` file directly, run it through the network, and plot only the prediction.

In [ ]:
def test_on_unlabeled_bin(bin_path):
    if not os.path.exists(bin_path):
        print(f"ERROR: Could not find {bin_path}.")
        return
        
    print(f"Loading unlabeled .bin file from {bin_path}...")
    scan = np.fromfile(bin_path, dtype=np.float32).reshape((-1, 4))
    
    # Remove NaNs just in case to prevent CUDA device-side asserts in PointNet++ Farthest Point Sampling!
    scan = scan[~np.isnan(scan).any(axis=1)]
    
    # PointNet++ requires exactly 4096 points for this architecture.
    num_raw = scan.shape[0]
    if num_raw >= 4096:
        choice = np.random.choice(num_raw, 4096, replace=False)
    else:
        choice = np.random.choice(num_raw, 4096, replace=True)
    sampled_scan = scan[choice, :]
    
    # Prepare input tensor: [Batch=1, Channels=4, Points=4096]
    point_features = torch.tensor(sampled_scan, dtype=torch.float32).transpose(0, 1)
    inputs = point_features.unsqueeze(0)
    xyz = inputs[0, :3, :].transpose(0, 1).numpy()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_outdoor_model(num_classes=20, input_channels=1)
    CHECKPOINT_PATH_10 = '/content/drive/MyDrive/checkpoints_kitti/semantickitti_epoch_10.pth'
    
    if os.path.exists(CHECKPOINT_PATH_10):
        model.load_state_dict(torch.load(CHECKPOINT_PATH_10, map_location=device, weights_only=False))
    else:
        print("ERROR: Checkpoint not found!")
        return
        
    model = model.to(device)
    model.eval()
    
    print("Running inference...")
    inputs = inputs.to(device)
    with torch.no_grad():
        predictions, _ = model(inputs)
    
    pred_labels = torch.argmax(predictions, dim=2).squeeze(0).cpu().numpy()
    
    name_map = get_semantickitti_names()
    unique_classes = np.unique(pred_labels)
    detected_objects = [name_map.get(c, "Unknown") for c in unique_classes if c != 0]
    print(f"\n---> Objects detected by PointNet++ in Sequence 20: {', '.join(detected_objects)}\n")
    
    color_map = get_semantickitti_colors()
    color_map[0] = [255, 255, 255] # Map unlabeled to white
    pred_colors = np.array([color_map.get(l, [255, 255, 255]) for l in pred_labels]) / 255.0
    
    plt.figure(figsize=(10, 10))
    plt.scatter(xyz[:, 0], xyz[:, 1], c=pred_colors, s=5, alpha=0.8)
    plt.title(f"Sequence 20 Prediction ({os.path.basename(bin_path)})", fontsize=16)
    plt.axis('equal')
    plt.gca().set_facecolor('black')
    plt.show()

test_on_unlabeled_bin('/content/dataset_trimmed/sequences/20/velodyne/000000.bin')


## Testing with custom `.pcd` file
This cell allows you to run your newly trained weights against any `.pcd` file (from outside the SemanticKITTI dataset). Just make sure you upload the `.pcd` file to Colab's local directory first!

In [ ]:
from pypcd4 import PointCloud
import numpy as np
import torch
import matplotlib.pyplot as plt

def test_on_pcd(pcd_path):
    if not os.path.exists(pcd_path):
        print(f"ERROR: Could not find {pcd_path}. Please upload it to Colab.")
        return
        
    print(f"Loading custom point cloud from {pcd_path}...")
    pc = PointCloud.from_path(pcd_path)
    
    try:
        scan = pc.numpy(['x', 'y', 'z', 'intensity'])
    except Exception as e:
        print(f"Warning: Could not find 'intensity' field. Defaulting to 0 intensity.")
        xyz = pc.numpy(['x', 'y', 'z'])
        scan = np.column_stack((xyz, np.zeros((xyz.shape[0], 1))))
    
    scan = scan.astype(np.float32)
    scan = scan[~np.isnan(scan).any(axis=1)]
    
    if np.max(scan[:, 3]) > 1.0:
        scan[:, 3] = scan[:, 3] / 255.0
    
    num_raw = scan.shape[0]
    if num_raw >= 4096:
        choice = np.random.choice(num_raw, 4096, replace=False)
    else:
        choice = np.random.choice(num_raw, 4096, replace=True)
    sampled_scan = scan[choice, :]
    
    point_features = torch.tensor(sampled_scan, dtype=torch.float32).transpose(0, 1)
    inputs = point_features.unsqueeze(0)
    xyz = inputs[0, :3, :].transpose(0, 1).numpy()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_outdoor_model(num_classes=20, input_channels=1)
    CHECKPOINT_PATH_10 = '/content/drive/MyDrive/checkpoints_kitti/semantickitti_epoch_10.pth'
    
    if os.path.exists(CHECKPOINT_PATH_10):
        model.load_state_dict(torch.load(CHECKPOINT_PATH_10, map_location=device, weights_only=False))
    else:
        print("ERROR: Checkpoint not found!")
        return
        
    model = model.to(device)
    model.eval()
    
    print("Running inference...")
    inputs = inputs.to(device)
    with torch.no_grad():
        predictions, _ = model(inputs)
    
    pred_labels = torch.argmax(predictions, dim=2).squeeze(0).cpu().numpy()
    
    name_map = get_semantickitti_names()
    unique_classes = np.unique(pred_labels)
    detected_objects = [name_map.get(c, "Unknown") for c in unique_classes if c != 0]
    
    if len(detected_objects) == 0:
        print(f"\n---> Warning: PointNet++ predicted 'unlabeled' for every single point in the frame!")
    else:
        print(f"\n---> Objects detected by PointNet++ in this custom frame: {', '.join(detected_objects)}\n")
    
    color_map = get_semantickitti_colors()
    color_map[0] = [255, 255, 255]
    pred_colors = np.array([color_map.get(l, [255, 255, 255]) for l in pred_labels]) / 255.0
    
    plt.figure(figsize=(10, 10))
    plt.scatter(xyz[:, 0], xyz[:, 1], c=pred_colors, s=5, alpha=0.8)
    plt.title("PointNet++ Custom PCD Prediction", fontsize=16)
    plt.axis('equal')
    plt.gca().set_facecolor('black')
    plt.show()

# Simply change 'frame_00010.pcd' to whatever you upload to Colab!
test_on_pcd('/content/frame_00010.pcd')
